In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Load all _total datasets and pick the best row per architecture × benchmark

In [ ]:
datasets = {
    "BatchTopK": "data/batch_total/results.parquet",
    "Matryoshka": "data/matryoshka_total/results.parquet",
    "MatchingPursuit": "data/mp_total/results.parquet",
    "Standard": "data/standard_total/results.parquet",
}

dfs = []
for arch, path in datasets.items():
    d = pd.read_parquet(path).reset_index()
    d["architecture"] = arch
    dfs.append(d)

raw = pd.concat(dfs, ignore_index=True)
raw = raw.rename(columns={"f1_score": "f1", "explained_variance": "r2"})
print(f"Total rows: {len(raw)}")

In [ ]:
metrics = ["f1", "mcc", "r2"]
cols = ["architecture", "benchmark", "sae_l0", "true_l0"] + metrics

# For each metric: pick the row with the highest score per (architecture, benchmark)
bests = {}
for metric in metrics:
    idx = raw.groupby(["architecture", "benchmark"])[metric].idxmax()
    bests[metric] = (
        raw.loc[idx, cols].copy().rename(columns={"sae_l0": f"sae_l0_at_best_{metric}"})
    )

# Merge into one comparison table keyed on (architecture, benchmark)
df = (
    bests["f1"]
    .merge(
        bests["mcc"][["architecture", "benchmark", "mcc", "sae_l0_at_best_mcc"]],
        on=["architecture", "benchmark"],
    )
    .merge(
        bests["r2"][["architecture", "benchmark", "r2", "sae_l0_at_best_r2"]],
        on=["architecture", "benchmark"],
    )
)

df = df.sort_values(["benchmark", "architecture"]).reset_index(drop=True)
df

## Best score heatmaps: benchmark × architecture

In [ ]:
arch_order = ["BatchTopK", "Matryoshka", "MatchingPursuit", "Standard"]
metric_labels = {"f1": "F1", "mcc": "MCC", "r2": "R²"}

fig = make_subplots(rows=1, cols=3, subplot_titles=[metric_labels[m] for m in metrics])

for i, metric in enumerate(metrics):
    pivot = df.pivot(index="benchmark", columns="architecture", values=metric)
    pivot = pivot.reindex(columns=arch_order)
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=pivot.columns.tolist(),
            y=pivot.index.tolist(),
            colorscale="Viridis",
            zmin=0,
            zmax=1,
            showscale=(i == 2),
            text=pivot.values.round(3),
            texttemplate="%{text}",
        ),
        row=1,
        col=i + 1,
    )

fig.update_layout(
    height=380,
    width=1100,
    title_text="Best F1 / MCC / R² per Architecture × Benchmark",
)
fig.show()

## Grouped bar: best scores by architecture (averaged across benchmarks)

In [ ]:
avg = df.groupby("architecture")[metrics].mean().reset_index()
melted = avg.melt(
    id_vars="architecture", value_vars=metrics, var_name="metric", value_name="score"
)
melted["metric"] = melted["metric"].map(metric_labels)

fig = px.bar(
    melted,
    x="architecture",
    y="score",
    color="metric",
    barmode="group",
    category_orders={"architecture": arch_order},
    labels={
        "score": "Score (avg across benchmarks)",
        "architecture": "Architecture",
        "metric": "Metric",
    },
    title="Best F1 / MCC / R² averaged across benchmarks",
    height=450,
    width=750,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_yaxes(range=[0, 1])
fig.show()

## SAE L0 at best F1 — where does each architecture peak?

In [ ]:
fig = px.strip(
    df,
    x="architecture",
    y="sae_l0_at_best_f1",
    color="benchmark",
    category_orders={"architecture": arch_order},
    labels={"sae_l0_at_best_f1": "SAE L0 at best F1", "architecture": "Architecture"},
    title="SAE L0 at which best F1 is achieved, by architecture and benchmark",
    height=450,
    width=750,
)
fig.add_hline(
    y=df["true_l0"].mean(),
    line_dash="dash",
    line_color="gray",
    annotation_text=f"avg true L0 ({df['true_l0'].mean():.2f})",
)
fig.show()

## Win counts: which architecture achieves the best score per benchmark?

In [ ]:
wins = {}
for metric in metrics:
    idx = df.groupby("benchmark")[metric].idxmax()
    wins[metric_labels[metric]] = df.loc[idx, "architecture"].value_counts()

wins_df = pd.DataFrame(wins).fillna(0).astype(int).reindex(arch_order).fillna(0)
print("Benchmark wins per architecture")
wins_df